# Chile Supply Chain: Full Diagnostic Print

Standalone script that loads all pipeline output CSVs and prints a comprehensive summary. No plots, no modifications, read-only.

Run after `Chile_MainAn.ipynb`. Covers:
1. File summary
2. Inventory breakdown
3. Copper production
4. Molybdenum production
5. Mine-to-plant links
6. Unified edge table
7. Downstream supply chain
8. Smelter connectivity
9. Port summary
10. Export destinations
11. Path traceability
12. Resource/reserve data coverage
13. Port distance comparison
14. Multi-match audit

In [1]:
# %% 0. Setup
import os
import numpy as np
import pandas as pd
from collections import Counter

BASE_DIR = "/Users/leoss/Desktop/GitHub/Capstone/Case studies/Chile"
PRELIM   = os.path.join(BASE_DIR, "Preliminary")

def header(title, width=70):
    print(f"\n{'=' * width}")
    print(f"  {title}")
    print(f"{'=' * width}")

def subheader(title):
    print(f"\n  --- {title} ---")

# ── Load all CSVs ────────────────────────────────────────────────────────
inv     = pd.read_csv(os.path.join(PRELIM, "Chile_Minerals_Inventory.csv"), low_memory=False)
links   = pd.read_csv(os.path.join(PRELIM, "Chile_Mine_Plant_Links.csv"))
edges   = pd.read_csv(os.path.join(PRELIM, "Chile_Supply_Chain_Edges.csv"))
ds      = pd.read_csv(os.path.join(PRELIM, "Chile_Downstream_Links.csv"))
exports = pd.read_csv(os.path.join(PRELIM, "Chile_Export_Destinations.csv"))
ports   = pd.read_csv(os.path.join(PRELIM, "Chile_Ports.csv"))

comm_col = "COMMODITY_LIST_STR" if "COMMODITY_LIST_STR" in inv.columns else "ALL_COMMODITIES_RAW"

print("=" * 70)
print("  CHILE MINERAL SUPPLY CHAIN: FULL DIAGNOSTIC REPORT")
print("=" * 70)


# ══════════════════════════════════════════════════════════════════════════
# 1. FILE SUMMARY
# ══════════════════════════════════════════════════════════════════════════

header("1. FILE SUMMARY")

files = {
    "Chile_Minerals_Inventory.csv":  inv,
    "Chile_Mine_Plant_Links.csv":    links,
    "Chile_Supply_Chain_Edges.csv":  edges,
    "Chile_Downstream_Links.csv":    ds,
    "Chile_Export_Destinations.csv": exports,
    "Chile_Ports.csv":              ports,
}
for name, df in files.items():
    print(f"  {name:<40} {len(df):>6} rows x {len(df.columns):>3} cols")

print(f"\n  Inventory columns: {', '.join(inv.columns[:10])}...")
print(f"  Edge columns:      {', '.join(edges.columns.tolist())}")


# ══════════════════════════════════════════════════════════════════════════
# 2. INVENTORY BREAKDOWN
# ══════════════════════════════════════════════════════════════════════════

header("2. INVENTORY BREAKDOWN")

subheader("Facility types")
for ft, count in inv["FACILITY_TYPE"].value_counts().items():
    print(f"    {ft:<40} {count:>5}")

if "CHAIN_STAGE" in inv.columns:
    subheader("Chain stages")
    for stage, count in inv["CHAIN_STAGE"].value_counts().items():
        print(f"    {stage:<20} {count:>5}")

subheader("Primary commodities (top 15)")
for comm, count in inv["PRIMARY_COMMODITY"].value_counts().head(15).items():
    print(f"    {str(comm):<25} {count:>5}")

if "SOURCE" in inv.columns:
    subheader("Data sources")
    for src, count in inv["SOURCE"].value_counts().items():
        print(f"    {str(src):<40} {count:>5}")

subheader("Coordinate coverage")
has_coords = inv["LATITUD"].notna() & inv["LONGITUD"].notna()
print(f"    With coordinates:    {has_coords.sum():>5} / {len(inv)} ({has_coords.mean()*100:.1f}%)")
print(f"    Missing coordinates: {(~has_coords).sum():>5}")

if "OPERATOR_NAME" in inv.columns:
    has_op = inv["OPERATOR_NAME"].notna() & (inv["OPERATOR_NAME"] != "")
    print(f"    With operator:       {has_op.sum():>5} / {len(inv)} ({has_op.mean()*100:.1f}%)")


# ══════════════════════════════════════════════════════════════════════════
# 3. COPPER PRODUCTION
# ══════════════════════════════════════════════════════════════════════════

header("3. COPPER PRODUCTION (COCHILCO 2024)")

cu = inv[inv["COCHILCO_CU_2024_KMT"].notna()].copy()
cu = cu.sort_values("COCHILCO_CU_2024_KMT", ascending=False)
cu_total = cu["COCHILCO_CU_2024_KMT"].sum()

subheader(f"All matched producers ({len(cu)} records, {cu_total:,.1f} kMT)")
for _, row in cu.iterrows():
    name = row["FACILITY_NAME"]
    prod = row["COCHILCO_CU_2024_KMT"]
    ftype = row["FACILITY_TYPE"]
    pct = prod / cu_total * 100
    company = row.get("COCHILCO_COMPANY", "")
    company_str = f"  [{company}]" if pd.notna(company) and company else ""
    print(f"    {name:<40} {prod:>8,.1f} kMT  ({pct:>5.1f}%)  {ftype}{company_str}")

subheader("Concentration")
cumulative = 0
for threshold in [1, 3, 5, 10, 15, 20]:
    if threshold <= len(cu):
        top_prod = cu.head(threshold)["COCHILCO_CU_2024_KMT"].sum()
        print(f"    Top {threshold:>2}: {top_prod:>8,.1f} kMT  ({top_prod/cu_total*100:>5.1f}%)")

subheader("Non-mine facilities with production")
non_mine = cu[~cu["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)]
if len(non_mine) > 0:
    for _, row in non_mine.iterrows():
        print(f"    {row['FACILITY_NAME']:<40} {row['FACILITY_TYPE']:<20} {row['COCHILCO_CU_2024_KMT']:>8.1f} kMT")
    print(f"    Total non-mine: {non_mine['COCHILCO_CU_2024_KMT'].sum():,.1f} kMT "
          f"({non_mine['COCHILCO_CU_2024_KMT'].sum()/cu_total*100:.1f}%)")
else:
    print("    None")


# ══════════════════════════════════════════════════════════════════════════
# 4. MOLYBDENUM PRODUCTION
# ══════════════════════════════════════════════════════════════════════════

header("4. MOLYBDENUM PRODUCTION (COCHILCO 2024)")

if "COCHILCO_MO_2024_MT" in inv.columns:
    mo = inv[inv["COCHILCO_MO_2024_MT"].notna()].sort_values("COCHILCO_MO_2024_MT", ascending=False)
    mo_total = mo["COCHILCO_MO_2024_MT"].sum()
    print(f"  {len(mo)} producers matched, {mo_total:,.1f} MT total\n")
    for _, row in mo.iterrows():
        print(f"    {row['FACILITY_NAME']:<40} {row['COCHILCO_MO_2024_MT']:>10,.1f} MT  "
              f"({row['COCHILCO_MO_2024_MT']/mo_total*100:>5.1f}%)")
else:
    print("  COCHILCO_MO_2024_MT column not found")


# ══════════════════════════════════════════════════════════════════════════
# 5. MINE-TO-PLANT LINKS
# ══════════════════════════════════════════════════════════════════════════

header("5. MINE-TO-PLANT LINKS")

print(f"  Total links: {len(links)}")

subheader("Product form distribution")
if "PRODUCT_FORM" in links.columns:
    for pf, count in links["PRODUCT_FORM"].value_counts().items():
        print(f"    {pf:<20} {count:>6}  ({count/len(links)*100:>5.1f}%)")

subheader("Commodity distribution")
if "SHARED_COMMODITIES" in links.columns:
    comm_counts = Counter()
    for comms in links["SHARED_COMMODITIES"].dropna():
        for c in str(comms).split(","):
            c = c.strip()
            if c:
                comm_counts[c] += 1
    for c, n in comm_counts.most_common():
        print(f"    {c:<20} {n:>6} links")

subheader("Distance statistics")
dist = pd.to_numeric(links["DISTANCE_KM"], errors="coerce").dropna()
print(f"    Links with distance: {len(dist)} / {len(links)}")
print(f"    Min:    {dist.min():>8.1f} km")
print(f"    P25:    {dist.quantile(0.25):>8.1f} km")
print(f"    Median: {dist.median():>8.1f} km")
print(f"    Mean:   {dist.mean():>8.1f} km")
print(f"    P75:    {dist.quantile(0.75):>8.1f} km")
print(f"    Max:    {dist.max():>8.1f} km")
print(f"    Std:    {dist.std():>8.1f} km")

subheader("Distance by product form")
if "PRODUCT_FORM" in links.columns:
    links["_dist"] = pd.to_numeric(links["DISTANCE_KM"], errors="coerce")
    for pf in links["PRODUCT_FORM"].unique():
        sub = links[links["PRODUCT_FORM"] == pf]["_dist"].dropna()
        if len(sub) > 0:
            print(f"    {pf:<20}  n={len(sub):<5}  median={sub.median():.0f} km  "
                  f"mean={sub.mean():.0f} km  max={sub.max():.0f} km")
    links.drop(columns=["_dist"], inplace=True, errors="ignore")

subheader("Operator match quality")
if "MINE_OPERATOR" in links.columns and "PLANT_OPERATOR" in links.columns:
    def classify_op(row):
        mine_op  = str(row.get("MINE_OPERATOR", "")).strip()
        plant_op = str(row.get("PLANT_OPERATOR", "")).strip()
        if not mine_op or mine_op == "nan":
            return "no_mine_operator"
        if not plant_op or plant_op == "nan":
            return "no_plant_operator"
        if mine_op.lower() == plant_op.lower():
            return "same_operator"
        if mine_op.lower()[:8] in plant_op.lower() or plant_op.lower()[:8] in mine_op.lower():
            return "partial_match"
        return "different_operator"

    links["_op"] = links.apply(classify_op, axis=1)
    for cat, sub in links.groupby("_op"):
        med = pd.to_numeric(sub["DISTANCE_KM"], errors="coerce").median()
        print(f"    {cat:<25} {len(sub):>6} ({len(sub)/len(links)*100:>5.1f}%)  "
              f"median dist: {med:.0f} km")

    cross = links[links["_op"] == "different_operator"]
    if len(cross) > 0:
        cross_dist = pd.to_numeric(cross["DISTANCE_KM"], errors="coerce")
        print(f"\n    Cross-company links by distance band:")
        for label, lo, hi in [("<20 km", 0, 20), ("20-50 km", 20, 50),
                               ("50-100 km", 50, 100), ("100-150 km", 100, 150),
                               ("150+ km", 150, 9999)]:
            n = len(cross_dist[(cross_dist >= lo) & (cross_dist < hi)])
            print(f"      {label:<15} {n:>5}")
    links.drop(columns=["_op"], inplace=True, errors="ignore")
else:
    print("    MINE_OPERATOR / PLANT_OPERATOR columns not found")


# ══════════════════════════════════════════════════════════════════════════
# 6. UNIFIED EDGE TABLE
# ══════════════════════════════════════════════════════════════════════════

header("6. UNIFIED EDGE TABLE")

print(f"  Total edges: {len(edges)}")

subheader("By edge type")
for et, count in edges["EDGE_TYPE"].value_counts().sort_index().items():
    print(f"    {et:<30} {count:>6}")

subheader("Edge matching methods")
methods = {
    "mine_to_plant":        "distance (haversine + shared commodity)",
    "concentrate_to_port":  "distance (nearest port), except Escondida->Coloso",
    "sxew_to_port":         "distance (nearest cathode-handling port)",
    "smelter_to_port":      "direct (hardcoded smelter->port mapping)",
    "port_to_country":      "direct (COCHILCO export tables + port share weights)",
}
for et in edges["EDGE_TYPE"].unique():
    n = len(edges[edges["EDGE_TYPE"] == et])
    method = methods.get(et, "unknown")
    print(f"    {et:<30} {n:>5} edges  [{method}]")


# ══════════════════════════════════════════════════════════════════════════
# 7. DOWNSTREAM SUPPLY CHAIN
# ══════════════════════════════════════════════════════════════════════════

header("7. DOWNSTREAM SUPPLY CHAIN")

print(f"  Downstream edges (non mine-to-plant): {len(ds)}")

subheader("By edge type")
for et, count in ds["EDGE_TYPE"].value_counts().items():
    print(f"    {et:<30} {count:>5}")

subheader("By commodity")
if "COMMODITIES" in ds.columns:
    for comm, count in ds["COMMODITIES"].value_counts().items():
        print(f"    {comm:<20} {count:>5} edges")

subheader("Mine-plant links by commodity with downstream status")
if "SHARED_COMMODITIES" in links.columns:
    comm_counts = Counter()
    for comms in links["SHARED_COMMODITIES"].dropna():
        for c in str(comms).split(","):
            c = c.strip()
            if c:
                comm_counts[c] += 1
    for c, n in comm_counts.most_common():
        has_ds = ("YES" if c == "Copper"
                  else "port_to_country only" if c in ["Molybdenum", "Lithium", "Iodine"]
                  else "NO")
        print(f"    {c:<20} {n:>5} links  downstream: {has_ds}")


# ══════════════════════════════════════════════════════════════════════════
# 8. SMELTER CONNECTIVITY
# ══════════════════════════════════════════════════════════════════════════

header("8. SMELTER CONNECTIVITY")

if "CHAIN_STAGE" in inv.columns:
    smelters = inv[inv["CHAIN_STAGE"] == "smelting"]["FACILITY_NAME"].unique()
    print(f"  Smelting facilities in inventory: {len(smelters)}\n")
    for name in sorted(smelters):
        m2p_in  = len(edges[(edges["EDGE_TYPE"] == "mine_to_plant") & (edges["TO_NAME"] == name)])
        c2s_in  = len(edges[(edges["EDGE_TYPE"] == "concentrate_to_smelter") & (edges["TO_NAME"] == name)])
        s2p_out = len(edges[(edges["EDGE_TYPE"] == "smelter_to_port") & (edges["FROM_NAME"] == name)])
        parts = []
        if m2p_in  > 0: parts.append(f"m2p_in={m2p_in}")
        if c2s_in  > 0: parts.append(f"c2s_in={c2s_in}")
        if s2p_out > 0: parts.append(f"s2p_out={s2p_out}")
        if not parts:    parts.append("DISCONNECTED")
        print(f"    {name:<62} [{', '.join(parts)}]")
else:
    print("  CHAIN_STAGE column not found")


# ══════════════════════════════════════════════════════════════════════════
# 9. PORT SUMMARY
# ══════════════════════════════════════════════════════════════════════════

header("9. PORT SUMMARY")

print(f"  Ports: {len(ports)}\n")
for _, p in ports.iterrows():
    print(f"    {p['name']:<25} {p['region']:<20} ({p['lat']:.2f}, {p['lon']:.2f})  "
          f"Products: {p['products']}")

subheader("Port inflows (domestic edges)")
port_edge_types = ["sxew_to_port", "concentrate_to_port", "smelter_to_port"]
port_inflows = edges[edges["EDGE_TYPE"].isin(port_edge_types)]
for et in port_edge_types:
    sub = port_inflows[port_inflows["EDGE_TYPE"] == et]
    port_list = sorted(sub["TO_NAME"].unique())
    print(f"    {et:<25} -> {', '.join(port_list)}")

subheader("Port outflows (export edges)")
p2c = edges[edges["EDGE_TYPE"] == "port_to_country"]
print(f"    {p2c['FROM_NAME'].nunique()} ports -> {p2c['TO_NAME'].nunique()} countries "
      f"({len(p2c)} edges)")
for port in sorted(p2c["FROM_NAME"].unique()):
    sub = p2c[p2c["FROM_NAME"] == port]
    print(f"    {port:<25} -> {sub['TO_NAME'].nunique()} countries ({len(sub)} edges)")


# ══════════════════════════════════════════════════════════════════════════
# 10. EXPORT DESTINATIONS
# ══════════════════════════════════════════════════════════════════════════

header("10. EXPORT DESTINATIONS")

print(f"  Total export edges: {len(exports)}\n")

for commodity in exports["COMMODITIES"].unique():
    sub = exports[exports["COMMODITIES"] == commodity]
    subheader(f"{commodity}")
    print(f"    Edges: {len(sub)}  |  Countries: {sub['TO_NAME'].nunique()}  |  "
          f"Ports: {sub['FROM_NAME'].nunique()}")

    if "DESTINATION_TOTAL" in sub.columns:
        by_country = sub.drop_duplicates(
            subset=["TO_NAME", "PRODUCT_FORM"]
        ).groupby("TO_NAME")["DESTINATION_TOTAL"].sum().sort_values(ascending=False)
        total_val = by_country.sum()
        unit = sub["EXPORT_UNIT"].iloc[0] if "EXPORT_UNIT" in sub.columns else ""

        print(f"    Total volume: {total_val:,.1f} {unit}")
        print(f"    Top destinations:")
        for country, val in by_country.head(10).items():
            pct = val / total_val * 100
            print(f"      {country:<25} {val:>10,.1f} {unit}  ({pct:>5.1f}%)")

    if "PRODUCT_FORM" in sub.columns:
        print(f"    By product form:")
        for pf, pf_sub in sub.groupby("PRODUCT_FORM"):
            n_countries = pf_sub["TO_NAME"].nunique()
            print(f"      {pf:<25} {len(pf_sub):>4} edges  ({n_countries} countries)")


# ══════════════════════════════════════════════════════════════════════════
# 11. PATH TRACEABILITY (mine -> port)
# ══════════════════════════════════════════════════════════════════════════

header("11. PATH TRACEABILITY")

cu_producers = inv[inv["COCHILCO_CU_2024_KMT"].notna() & (inv["COCHILCO_CU_2024_KMT"] > 0)]
cu_total_matched = cu_producers["COCHILCO_CU_2024_KMT"].sum()
connected_prod = 0
disconnected = []

for _, mrow in cu_producers.iterrows():
    mine_name = mrow["FACILITY_NAME"]
    stage = mrow.get("CHAIN_STAGE", "extraction")

    # Non-extraction facilities (SX-EW plants with production)
    if stage != "extraction":
        direct_ds = edges[
            (edges["FROM_NAME"] == mine_name) &
            (edges["EDGE_TYPE"].isin(["sxew_to_port", "smelter_to_port", "concentrate_to_port"]))
        ]
        if len(direct_ds) > 0:
            connected_prod += mrow["COCHILCO_CU_2024_KMT"]
        else:
            disconnected.append((mine_name, mrow["COCHILCO_CU_2024_KMT"]))
        continue

    mine_edges = edges[(edges["EDGE_TYPE"] == "mine_to_plant") & (edges["FROM_NAME"] == mine_name)]
    if len(mine_edges) == 0:
        mine_edges = edges[
            (edges["EDGE_TYPE"] == "mine_to_plant") &
            edges["FROM_NAME"].str.contains(mine_name[:8], case=False, na=False, regex=False)
        ]

    if len(mine_edges) == 0:
        disconnected.append((mine_name, mrow["COCHILCO_CU_2024_KMT"]))
        continue

    plants = set(mine_edges["TO_NAME"])
    reaches_port = False
    for plant in plants:
        ds_types = ["concentrate_to_port", "sxew_to_port", "smelter_to_port"]
        if edges[(edges["FROM_NAME"] == plant) & (edges["EDGE_TYPE"].isin(ds_types))].shape[0] > 0:
            reaches_port = True
            break
        # Check via smelter
        sm_edges = edges[(edges["FROM_NAME"] == plant) & (edges["EDGE_TYPE"] == "concentrate_to_smelter")]
        for _, se in sm_edges.iterrows():
            if edges[(edges["FROM_NAME"] == se["TO_NAME"]) & (edges["EDGE_TYPE"] == "smelter_to_port")].shape[0] > 0:
                reaches_port = True
                break
        if reaches_port:
            break

    if reaches_port:
        connected_prod += mrow["COCHILCO_CU_2024_KMT"]
    else:
        disconnected.append((mine_name, mrow["COCHILCO_CU_2024_KMT"]))

n_connected = len(cu_producers) - len(disconnected)
print(f"  Cu producers: {len(cu_producers)}")
print(f"  Reaching port: {n_connected} / {len(cu_producers)}")
print(f"  Production reaching port: {connected_prod:,.1f} / {cu_total_matched:,.1f} kMT "
      f"({connected_prod/cu_total_matched*100:.1f}%)")

if disconnected:
    print(f"\n  Disconnected ({len(disconnected)}):")
    for name, prod in sorted(disconnected, key=lambda x: -x[1]):
        print(f"    {name:<45} {prod:>8.1f} kMT")
else:
    print(f"\n  All producers connected to a port.")


# ══════════════════════════════════════════════════════════════════════════
# 12. RESOURCE / RESERVE DATA COVERAGE
# ══════════════════════════════════════════════════════════════════════════

header("12. RESOURCE / RESERVE DATA COVERAGE")

EXTRACTION_TYPES = ["Mine (active)", "Mine (idle)", "Prospect/Project", "Mine (USGS)"]
extraction = inv[inv["FACILITY_TYPE"].isin(EXTRACTION_TYPES)].copy()
res_cols = sorted([c for c in inv.columns if c.endswith(("_Resource", "_Reserve"))])

extraction["has_any"] = extraction[res_cols].notna().any(axis=1)
n_has = extraction["has_any"].sum()
n_total = len(extraction)
print(f"  Extraction facilities: {n_total}")
print(f"  With any resource/reserve data: {n_has} ({n_has/n_total*100:.1f}%)")
print(f"  Missing all resource/reserve:   {n_total - n_has} ({(n_total-n_has)/n_total*100:.1f}%)")

subheader("By facility type")
for ft, sub in extraction.groupby("FACILITY_TYPE"):
    n = sub["has_any"].sum()
    print(f"    {ft:<25} {n:>4} / {len(sub):<4} ({n/len(sub)*100:>5.1f}%)")

subheader("By primary commodity (top 10)")
for comm in extraction["PRIMARY_COMMODITY"].value_counts().head(10).index:
    sub = extraction[extraction["PRIMARY_COMMODITY"] == comm]
    res_c, rev_c = f"{comm}_Resource", f"{comm}_Reserve"
    n_res = sub[res_c].notna().sum() if res_c in sub.columns else 0
    n_rev = sub[rev_c].notna().sum() if rev_c in sub.columns else 0
    n_any = sub.apply(
        lambda r: pd.notna(r.get(res_c)) or pd.notna(r.get(rev_c)), axis=1
    ).sum() if (res_c in sub.columns or rev_c in sub.columns) else 0
    print(f"    {comm:<20} {n_any:>3} / {len(sub):<3} ({n_any/len(sub)*100:>5.1f}%)  "
          f"[resource: {n_res}, reserve: {n_rev}]")


# ══════════════════════════════════════════════════════════════════════════
# 13. PORT DISTANCE COMPARISON (if available)
# ══════════════════════════════════════════════════════════════════════════

header("13. PORT DISTANCE COMPARISON")

comp_path = os.path.join(PRELIM, "Port_Distance_Comparison.csv")
if os.path.exists(comp_path):
    comp = pd.read_csv(comp_path)
    for product in comp["product"].unique():
        sub = comp[comp["product"] == product].sort_values("actual_share", ascending=False)
        print(f"\n  {product.upper()}")
        print(f"  {'-'*72}")
        for _, row in sub.iterrows():
            actual = row["actual_share"] * 100
            simulated = row["simulated_share"] * 100
            diff = row["difference"] * 100
            if abs(diff) < 2:
                indicator = "="
            elif diff > 0:
                indicator = "^"
            else:
                indicator = "v"
            print(f"    {row['port']:<25} Actual: {actual:>5.1f}%  |  "
                  f"Optimal: {simulated:>5.1f}%  |  D: {diff:>+6.1f}% {indicator}")
        mad = sub["difference"].abs().mean() * 100
        tvol = sub["difference"].abs().sum() / 2 * 100
        print(f"    Mean absolute diff: {mad:.1f}%  |  Total volume mismatch: {tvol:.1f}%")
else:
    print("  Port_Distance_Comparison.csv not found (run Section 7 of pipeline)")


# ══════════════════════════════════════════════════════════════════════════
# 14. MULTI-MATCH AUDIT
# ══════════════════════════════════════════════════════════════════════════

header("14. MULTI-MATCH AUDIT")

COMPANY_TO_DEPOSIT = {
    "División El Teniente": ["El Teniente"],
    "División Chuquicamata": ["Chuquicamata"],
    "División Radomiro Tomic": ["Radomiro Tomic"],
    "División Andina": ["Andina"],
    "División Ministro Hales": ["Ministro Hales"],
    "División Gabriela Mistral": ["Gabriela Mistral"],
    "División Salvador": ["Salvador"],
    "Escondida": ["Escondida"], "Collahuasi": ["Collahuasi"],
    "Los Pelambres": ["Los Pelambres"], "Spence": ["Spence"],
    "Quebrada Blanca": ["Quebrada Blanca"],
    "Los Bronces": ["Los Bronces"], "Sierra Gorda": ["Sierra Gorda"],
    "Candelaria": ["Candelaria"], "Caserones": ["Caserones"],
    "Centinela (Sulfuros)": ["Centinela"],
    "El Abra": ["El Abra"], "Zaldivar": ["Zaldivar", "Zaldívar"],
    "Antucoya": ["Antucoya"], "Lomas Bayas": ["Lomas Bayas"],
    "Mantoverde": ["Mantoverde"], "El Soldado": ["El Soldado"],
    "Mantos Blancos": ["Mantos Blancos"], "Andacollo": ["Andacollo"],
    "Michilla": ["Michilla"], "Franke": ["Franke"],
    "Mantos de la Luna": ["Mantos de la Luna"],
    "Cerro Negro": ["Cerro Negro"], "Cerro Colorado": ["Cerro Colorado"],
    "Las Cenizas": ["Las Cenizas", "Las Luces"],
    "Tres Valles": ["Tres Valles"],
}

multi_count = 0
for company, search_terms in COMPANY_TO_DEPOSIT.items():
    for term in search_terms:
        matches = inv[inv["FACILITY_NAME"].str.contains(term, case=False, na=False, regex=False)]
        if len(matches) > 1:
            mine_matches = matches[matches["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)]
            other = matches[~matches["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)]
            multi_count += 1
            print(f"  {company} ('{term}'): {len(matches)} matches "
                  f"({len(mine_matches)} mines, {len(other)} non-mines)")
            for _, row in matches.iterrows():
                selected = ""
                if len(mine_matches) > 0 and row.name == mine_matches.index[0]:
                    selected = " <-- SELECTED"
                elif len(mine_matches) == 0 and row.name == matches.index[0]:
                    selected = " <-- SELECTED"
                print(f"    {row['FACILITY_NAME']:<50} {row['FACILITY_TYPE']:<20}{selected}")
        if len(matches) > 0:
            break

print(f"\n  Companies with multiple matches: {multi_count}")


# ══════════════════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════════════════

header("SUMMARY")

print(f"  Inventory:           {len(inv)} facilities")
print(f"  Mine-plant links:    {len(links)} links")
print(f"  Supply chain edges:  {len(edges)} total edges")
print(f"    mine_to_plant:     {len(edges[edges['EDGE_TYPE']=='mine_to_plant'])}")
print(f"    downstream:        {len(ds)} edges")
print(f"    port_to_country:   {len(edges[edges['EDGE_TYPE']=='port_to_country'])}")
print(f"  Export destinations: {len(exports)} edges -> {exports['TO_NAME'].nunique()} countries")
print(f"  Ports:               {len(ports)}")
print(f"  Cu matched:          {len(cu)} producers, {cu_total:,.1f} kMT")
if "COCHILCO_MO_2024_MT" in inv.columns:
    print(f"  Mo matched:          {len(mo)} producers, {mo_total:,.1f} MT")
print(f"  Path traceability:   {n_connected}/{len(cu_producers)} mines reach port "
      f"({connected_prod:,.1f}/{cu_total_matched:,.1f} kMT)")
res_pct = n_has / n_total * 100
print(f"  Resource coverage:   {n_has}/{n_total} extraction sites ({res_pct:.1f}%)")
print(f"\n{'=' * 70}")
print("  REPORT COMPLETE")
print(f"{'=' * 70}")


  CHILE MINERAL SUPPLY CHAIN: FULL DIAGNOSTIC REPORT

  1. FILE SUMMARY
  Chile_Minerals_Inventory.csv                460 rows x  96 cols
  Chile_Mine_Plant_Links.csv                 1041 rows x  21 cols
  Chile_Supply_Chain_Edges.csv               1416 rows x  12 cols
  Chile_Downstream_Links.csv                  375 rows x  12 cols
  Chile_Export_Destinations.csv               561 rows x  15 cols
  Chile_Ports.csv                              11 rows x   6 cols

  Inventory columns: Nitrate_Resource, COMMTYPE, LONGITUD, COMBINACION_CRITICO, CRITICO_3, Iodine_Resource, ESTADO_DEPOSITO, Molybdenum_Reserve, OPERATOR_NAME, ECAP...
  Edge columns:      FROM_NAME, FROM_TYPE, FROM_LAT, FROM_LON, TO_NAME, TO_TYPE, TO_LAT, TO_LON, EDGE_TYPE, PRODUCT_FORM, COMMODITIES, DISTANCE_KM

  2. INVENTORY BREAKDOWN

  --- Facility types ---
    Processing Plant                           123
    Prospect/Project                           113
    Mine (idle)                                 80
    Mine (a